# Dicta Libre: piloto sintético y selección de voz A

Ejecutar en Colab con GPU, por orden. Incluye piloto original, comparación A/B y lote completo A. No contiene audios ni resultados privados. La referencia es la muestra pública es-MX de Resemble. Las salidas necesitan revisión humana; preferir A en tres casos no valida todo el corpus. Ver docs/CHATTERBOX_COLAB_PILOT.md.


In [ ]:
# Dicta Libre | piloto sintético, sin datos de pacientes
# Instalación aislada: conserva las dependencias del notebook de Colab.
import sys, subprocess, pathlib, time, json
ROOT = pathlib.Path('/content/dictalibre_pilot')
ROOT.mkdir(exist_ok=True)
PY = '/content/dictalibre_venv/bin/python'
SOURCE_COMMIT = '5de7a54aa4e5e2baadb0182dde554908b48b85c2'
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv'], capture_output=True, text=True).stdout)
print('Python Colab:', sys.version)
with (ROOT/'bootstrap.log').open('w') as log:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'virtualenv'], stdout=log, stderr=subprocess.STDOUT, check=True)
    subprocess.run([sys.executable, '-m', 'virtualenv', '/content/dictalibre_venv'], stdout=log, stderr=subprocess.STDOUT, check=True)
t0 = time.perf_counter()
with (ROOT/'installation.log').open('w') as log:
    result = subprocess.run([PY,'-m','pip','install','git+https://github.com/resemble-ai/chatterbox.git@'+SOURCE_COMMIT], stdout=log, stderr=subprocess.STDOUT)
(ROOT/'installation.json').write_text(json.dumps({'source_commit':SOURCE_COMMIT,'elapsed_s':time.perf_counter()-t0,'returncode':result.returncode}))
print((ROOT/'installation.log').read_text()[-6000:])
assert result.returncode == 0, 'Instalación fallida: revisar installation.log'
print('INSTALACION_OK')


In [ ]:
# Ejecutar tras INSTALACION_OK. Cada intento conserva sus resultados.
RUNNER = "import pathlib, json, csv, time, datetime, platform, subprocess, hashlib, random, traceback\nimport importlib.metadata as md\nimport numpy as np\nimport torch\nimport soundfile as sf\nfrom huggingface_hub import snapshot_download\nfrom chatterbox.mtl_tts import ChatterboxMultilingualTTS\nROOT = pathlib.Path('/content/dictalibre_pilot')\nRUN = ROOT / datetime.datetime.now(datetime.timezone.utc).strftime('run_%Y%m%dT%H%M%SZ')\nRUN.mkdir(parents=True, exist_ok=False)\ndef save(name, obj):\n    (RUN/name).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')\ntexts = [\n'Hallazgos dos puntos. Nódulo sólido de ocho milímetros en el lóbulo superior derecho. Punto.',\n'La lesión presenta una atenuación de menos diez unidades Hounsfield. Punto.',\n'Bronquiectasias cilíndricas en ambos lóbulos inferiores. Punto.',\n'Quiste renal izquierdo de dos coma cinco centímetros. Punto.',\n'No se observa derrame pleural ni neumotórax. Punto.',\n'Opacidad en vidrio esmerilado en el segmento apical del lóbulo superior izquierdo. Punto.',\n'Adenopatía mediastínica de doce por ocho milímetros. Punto.',\n'Colelitiasis sin signos de colecistitis aguda. Punto.',\n'Espondilolistesis de grado uno en el nivel L cinco S uno. Punto.',\n'Engrosamiento de la pared del íleon terminal. Punto.',\n'Conclusión dos puntos. Sin evidencia de tromboembolismo pulmonar. Punto y aparte.',\n'Se sugiere comparar con estudios previos, coma, si están disponibles. Punto final.'\n]\nconfig = dict(language_id='es', exaggeration=0.5, cfg_weight=0.5, temperature=0.8, repetition_penalty=1.2, min_p=0.05, top_p=1.0)\nsave('protocol.json', {'hypothesis':'V3 puede producir 12 WAV íntegros; pronunciación apta requiere revisión humana.', 'n_scripts':12,'repetitions_per_script':1,'voice':'builtin conds.pt; sin referencia personal','audio_source':'TTS sintético','condition':'limpio, sin ruido añadido','configuration':config,'seed_base':20261103,'interval':'CUDA sincronizada inmediatamente antes de generate hasta sincronización posterior; excluye escritura WAV, carga y descarga del modelo','metrics':{'generation_s':'segundos directos perf_counter','audio_duration_s':'muestras / frecuencia','rtf':'generation_s / audio_duration_s'},'acceptance':'Técnico: 12/12 WAV finitos, no silenciosos, legibles; auditivo: 12/12 conservan términos, cifras, negaciones y comandos, sin omisiones/repeticiones. No escalar antes de revisión humana.','human_review':None,'comparison':None,'network':None,'browser_version':None,'client_device':None,'instrumentation_overhead_s':None,'overhead_note':'Sincronización CUDA puede añadir sobrecosto no aislado; guardado y hash fuera del intervalo. No es benchmark comparativo.','clock_resolution_s':time.get_clock_info('perf_counter').resolution,'dictalibre_commit':None})\nassert torch.cuda.is_available(), 'GPU no disponible'\nmeta = {'started_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'python':platform.python_version(),'os':platform.platform(),'gpu':torch.cuda.get_device_name(0),'cuda':torch.version.cuda,'source_commit':'5de7a54aa4e5e2baadb0182dde554908b48b85c2','versions':{p:md.version(p) for p in ['chatterbox-tts','torch','torchaudio','transformers','numpy','soundfile']}}\nsave('environment.json',meta)\n(RUN/'pip-freeze.txt').write_text(subprocess.check_output([__import__('sys').executable,'-m','pip','freeze'], text=True))\nprint('RUN', RUN, flush=True)\nprint('Descargando checkpoint V3...', flush=True)\nt0=time.perf_counter()\ncheckpoint=snapshot_download('ResembleAI/chatterbox',allow_patterns=['ve.pt','t3_mtl23ls_v3.safetensors','s3gen.pt','grapheme_mtl_merged_expanded_v1.json','conds.pt','Cangjie5_TC.json'])\nmeta['model_revision']=pathlib.Path(checkpoint).name\nmeta['download_s']=time.perf_counter()-t0\nsave('environment.json',meta)\nt0=time.perf_counter()\nmodel=ChatterboxMultilingualTTS.from_local(checkpoint,device='cuda',t3_model='v3')\ntorch.cuda.synchronize()\nmeta['model_load_s']=time.perf_counter()-t0\nsave('environment.json',meta)\nrows=[]\nfor i,text in enumerate(texts,1):\n    row={'case_id':f'p{i:02d}','text':text,'seed':20261103+i,'repetition':1,'status':'error','generation_s':None,'duration_s':None,'rtf':None,'sample_rate':None,'sha256':None,'error':None,'human_review':None}\n    try:\n        random.seed(row['seed']); np.random.seed(row['seed']); torch.manual_seed(row['seed']); torch.cuda.manual_seed_all(row['seed'])\n        torch.cuda.synchronize(); start=time.perf_counter()\n        wav=model.generate(text,**config)\n        torch.cuda.synchronize(); row['generation_s']=time.perf_counter()-start\n        samples=wav.detach().cpu().numpy().squeeze()\n        assert samples.ndim==1 and samples.size>0 and np.isfinite(samples).all() and np.max(np.abs(samples))>0\n        path=RUN/(row['case_id']+'.wav')\n        sf.write(path,samples,model.sr,subtype='FLOAT')\n        check,sr=sf.read(path)\n        assert sr==model.sr and len(check)==len(samples) and np.isfinite(check).all()\n        row.update(status='ok',duration_s=len(samples)/sr,sample_rate=sr,sha256=hashlib.sha256(path.read_bytes()).hexdigest(),peak=float(np.max(np.abs(samples))))\n        row['rtf']=row['generation_s']/row['duration_s']\n    except Exception as exc:\n        row['error']=str(exc)\n        traceback.print_exc()\n    rows.append(row)\n    save('manifest.json',rows)\n    keys=sorted(set().union(*(r.keys() for r in rows)))\n    with (RUN/'manifest.csv').open('w',newline='',encoding='utf-8') as f:\n        writer=csv.DictWriter(f,fieldnames=keys); writer.writeheader(); writer.writerows(rows)\n    print(row['case_id'],row['status'],'duration_s=',row['duration_s'],'generation_s=',row['generation_s'],flush=True)\nsave('summary.json',{'completed_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'successful':sum(r['status']=='ok' for r in rows),'attempted':len(rows),'human_acceptance':None})\nprint('PILOTO_TERMINADO',sum(r['status']=='ok' for r in rows),'/12',flush=True)\n"
(ROOT/'generate_pilot.py').write_text(RUNNER, encoding='utf-8')
result = subprocess.run([PY, '-u', str(ROOT/'generate_pilot.py')])
print('Proceso terminado, código:', result.returncode)


In [ ]:
# Exporta originales y copias PCM16 con ganancia documentada.
import pathlib, shutil, json, subprocess
from google.colab import files
from IPython.display import Audio, display
ROOT = pathlib.Path('/content/dictalibre_pilot')
PY = '/content/dictalibre_venv/bin/python'
subprocess.run([PY, '-c', "import pathlib,json,hashlib\nimport numpy as np\nimport soundfile as sf\nroot=pathlib.Path('/content/dictalibre_pilot')\nfor run in root.glob('run_*'):\n    out=run/'lab_pcm16'; out.mkdir(exist_ok=True)\n    conversions=[]\n    for src in sorted(run.glob('*.wav')):\n        x,sr=sf.read(src); peak=float(np.max(np.abs(x)))\n        gain=min(1.0,0.98/peak)\n        dst=out/src.name\n        sf.write(dst,x*gain,sr,subtype='PCM_16')\n        y,ysr=sf.read(dst)\n        assert ysr==sr and len(y)==len(x) and np.isfinite(y).all() and np.max(np.abs(y))<=0.981\n        conversions.append({'file':src.name,'source_sha256':hashlib.sha256(src.read_bytes()).hexdigest(),'output_sha256':hashlib.sha256(dst.read_bytes()).hexdigest(),'source_peak':peak,'gain':gain,'output_peak':float(np.max(np.abs(y))),'format':'PCM_16','sample_rate':sr,'samples':len(y)})\n    (out/'conversion.json').write_text(json.dumps(conversions,indent=2))\n"],check=True)
archive = shutil.make_archive('/content/dictalibre_piloto_12', 'zip', ROOT)
for summary in sorted(ROOT.glob('run_*/summary.json')):
    print(summary.parent.name, summary.read_text())
for wav in sorted(ROOT.glob('run_*/lab_pcm16/*.wav')):
    print(wav.name)
    display(Audio(filename=str(wav), normalize=False))
files.download(archive)


In [ ]:
# Candidatos: mismo V3 general, referencia española oficial; NO es el checkpoint LatAm.
import pathlib, subprocess, urllib.request, hashlib, json, ast
ROOT=pathlib.Path('/content/dictalibre_pilot')
OUT=pathlib.Path('/content/dictalibre_pronunciacion')
OUT.mkdir(exist_ok=True)
url='https://storage.googleapis.com/chatterbox-demo-samples/mtl-v3-single-language-prompts/es-latam/es_mx_f1.wav'
ref=OUT/'referencia_oficial_es_mx.wav'
urllib.request.urlretrieve(url,ref)
protocol={'hypothesis':'Una referencia hablada en español reduce el acento anglófono percibido; un ajuste adicional puede reducir tropiezos.', 'baseline':'run_20261103T033052Z: rechazo cualitativo comunicado por usuario; puntuaciones ausentes', 'cases':['p02','p03','p09'], 'repetitions':1, 'conditions':'Mismo V3 general, texto y semilla por caso. Ambas variantes usan la referencia oficial es-MX. B cambia conjuntamente expresividad, CFG y temperatura: no permite atribuir causalidad a un parámetro.', 'metrics':{'naturalness':'valoración humana 1-5 por muestra','pronunciation_errors':'número de términos mal pronunciados por muestra','stumbles':'número de repeticiones o cortes involuntarios por muestra'}, 'interval':'Escuchar cada WAV completo una vez, inicio a fin; valoración después de la reproducción.', 'acceptance':'Cada muestra aceptada: naturalidad >=4/5, cero errores en cifras, negaciones y términos y cero tropiezos. Elegir variante solo tras escucha; no extrapolar a hablantes reales.', 'human_results':None, 'network':None, 'browser_version':None, 'reference_url':url,'reference_sha256':hashlib.sha256(ref.read_bytes()).hexdigest(), 'dictalibre_commit':None}
(OUT/'comparison_protocol.json').write_text(json.dumps(protocol,ensure_ascii=False,indent=2))
source=(ROOT/'generate_pilot.py').read_text()
source=source.replace("pathlib.Path('/content/dictalibre_pilot')", "pathlib.Path('/content/dictalibre_pronunciacion')")
source=source.replace("for i,text in enumerate(texts,1):", "for i in [2,3,9]:\n    text=texts[i-1]")
source=source.replace("'n_scripts':12", "'n_scripts':3").replace("'/12'", "'/3'")
source=source.replace('12 WAV','3 WAV').replace('12/12','3/3').replace('builtin conds.pt; sin referencia personal','Referencia oficial es-MX; ver comparison_protocol.json')
source=source.replace("snapshot_download('ResembleAI/chatterbox',allow_patterns=", "snapshot_download('ResembleAI/chatterbox',revision='5bb1f6ee58e50c3b8d408bc82a6d3740c2db6e18',allow_patterns=")
for variant,ex,cfg,temp in [('A_referencia_es',0.5,0.5,0.8),('B_referencia_es_suave',0.35,0.3,0.65)]:
    candidate=source.replace("'run_%Y%m%dT%H%M%SZ'",repr(variant+'_%Y%m%dT%H%M%SZ'))
    candidate=candidate.replace("exaggeration=0.5, cfg_weight=0.5, temperature=0.8",f"audio_prompt_path={str(ref)!r}, exaggeration={ex}, cfg_weight={cfg}, temperature={temp}")
    compile(candidate,'candidate','exec')
    script=OUT/(variant+'.py'); script.write_text(candidate,encoding='utf-8')
    print('GENERANDO',variant,flush=True)
    with (OUT/(variant+'.log')).open('w') as log:
        result=subprocess.run(['/content/dictalibre_venv/bin/python','-u',str(script)],stdout=log,stderr=subprocess.STDOUT)
    print((OUT/(variant+'.log')).read_text()[-2500:],flush=True)
    assert result.returncode==0, 'Revisar log: '+variant
print('COMPARACION_GENERADA')


In [ ]:
import pathlib, subprocess, shutil, json
from google.colab import files
from IPython.display import Audio, display
OUT=pathlib.Path('/content/dictalibre_pronunciacion')
conversion="""import pathlib,json,hashlib,numpy as np,soundfile as sf
root=pathlib.Path('/content/dictalibre_pronunciacion')
rows=[]
for src in sorted(root.glob('*_20*/*.wav')):
    x,sr=sf.read(src); peak=float(np.max(np.abs(x))); gain=min(1.,.98/peak)
    dst=src.parent/'pcm16'/src.name; dst.parent.mkdir(exist_ok=True)
    sf.write(dst,x*gain,sr,subtype='PCM_16')
    y,ysr=sf.read(dst); assert ysr==sr and len(x)==len(y) and np.isfinite(y).all()
    rows.append(dict(file=str(dst.relative_to(root)),sha256=hashlib.sha256(dst.read_bytes()).hexdigest(),gain=gain,sample_rate=sr,duration_s=len(y)/sr))
assert len(rows)==6, rows
(root/'conversion.json').write_text(json.dumps(rows,indent=2))
print(json.dumps(rows,indent=2))
"""
subprocess.run(['/content/dictalibre_venv/bin/python','-c',conversion],check=True)
for wav in sorted(OUT.glob('*_20*/pcm16/*.wav')):
    print(wav.parent.parent.name, wav.name)
    display(Audio(filename=str(wav),normalize=False))
archive=shutil.make_archive('/content/dictalibre_pronunciacion_AB','zip',OUT)
print('SEIS_CANDIDATOS_LISTOS',archive)
files.download(archive)


In [ ]:
# Voz A elegida por el usuario: 12 guiones, una toma por guion.
import pathlib, json, hashlib, subprocess, datetime, shutil
root=pathlib.Path('/content/dictalibre_voz_A_12')
root.mkdir(exist_ok=True)
ref=pathlib.Path('/content/dictalibre_pronunciacion/referencia_oficial_es_mx.wav')
assert hashlib.sha256(ref.read_bytes()).hexdigest()=='ee61089324b429371ae5562e1070283b34e220e742b60bff2457569f74484be4','Referencia distinta de la seleccionada'
source=pathlib.Path('/content/dictalibre_pilot/generate_pilot.py').read_text()
source=source.replace("pathlib.Path('/content/dictalibre_pilot')",repr(root).replace('PosixPath','pathlib.Path'))
source=source.replace("snapshot_download('ResembleAI/chatterbox',allow_patterns=", "snapshot_download('ResembleAI/chatterbox',revision='5bb1f6ee58e50c3b8d408bc82a6d3740c2db6e18',allow_patterns=")
source=source.replace("dict(language_id='es',",f"dict(audio_prompt_path={str(ref)!r}, language_id='es',")
source=source.replace('builtin conds.pt; sin referencia personal','Voz A: referencia oficial es-MX; SHA256 ee61089324b429371ae5562e1070283b34e220e742b60bff2457569f74484be4')
source=source.replace("'comparison':None", "'comparison':None,'selection_evidence':'Usuario prefirió A en p02/p03/p09; esta ejecución completa sigue pendiente de revisión','human_metrics':{'naturalness_1_to_5':None,'pronunciation_errors':None,'stumbles':None},'human_interval':'Escucha completa de cada WAV, inicio a fin; valoración después','human_acceptance_rule':'12/12 con naturalidad >=4/5, cero errores de términos, cifras, negaciones y comandos y cero tropiezos'")
source=source.replace("assert torch.cuda.is_available()", "save('generator.json', {'sha256': hashlib.sha256(pathlib.Path(__file__).read_bytes()).hexdigest()})\nassert torch.cuda.is_available()")
compile(source,'voice_A_12','exec')
script=root/('generate_A_'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'.py')
script.write_text(source,encoding='utf-8')
logpath=script.with_suffix('.log')
print('GENERANDO_12_VOZ_A',flush=True)
with logpath.open('w') as log:
    result=subprocess.run(['/content/dictalibre_venv/bin/python','-u',str(script)],stdout=log,stderr=subprocess.STDOUT)
print(logpath.read_text()[-3000:],flush=True)
assert result.returncode==0, 'Revisar log de generación'
run=max(root.glob('run_*'),key=lambda p:p.name)
summary=json.loads((run/'summary.json').read_text())
assert summary['successful']==12, summary
shutil.copy2(ref,run/'reference_es_mx.wav')
shutil.copy2(script,run/'generate.py')
print('DOCE_VOZ_A_GENERADOS',run,summary,flush=True)


In [ ]:
# Copias PCM16, pista completa para revisión y ZIP de esta ejecución.
import pathlib, json, subprocess, shutil
from IPython.display import Audio, display
from google.colab import files
root=pathlib.Path('/content/dictalibre_voz_A_12')
run=max(root.glob('run_*'),key=lambda p:p.name)
assert json.loads((run/'summary.json').read_text())['successful']==12
convert="""import pathlib,json,hashlib,numpy as np,soundfile as sf,sys
run=pathlib.Path(sys.argv[1]); out=run/'pcm16'; out.mkdir(exist_ok=True)
rows=[]; parts=[]; cursor=0
for case in json.loads((run/'manifest.json').read_text()):
    src=run/(case['case_id']+'.wav'); x,sr=sf.read(src)
    assert hashlib.sha256(src.read_bytes()).hexdigest()==case['sha256']
    assert x.ndim==1 and len(x)>0 and np.isfinite(x).all()
    gain=min(1.,.98/float(np.max(np.abs(x))))
    dst=out/src.name; sf.write(dst,x*gain,sr,subtype='PCM_16')
    y,ysr=sf.read(dst); assert ysr==sr==24000 and len(x)==len(y) and np.max(np.abs(y))<=.981
    rows.append(dict(case_id=case['case_id'],text=case['text'],file=str(dst.relative_to(run)),sha256=hashlib.sha256(dst.read_bytes()).hexdigest(),source_sha256=case['sha256'],gain=gain,sample_rate=sr,samples=len(y),duration_s=len(y)/sr,review_start_s=cursor/sr,human_acceptance=None))
    parts.append(y); cursor+=len(y)
    if len(rows)<12: parts.append(np.zeros(sr)); cursor+=sr
assert len(rows)==12
sf.write(run/'escucha_12_A.wav',np.concatenate(parts),24000,subtype='PCM_16')
(run/'conversion.json').write_text(json.dumps(rows,ensure_ascii=False,indent=2))
print('PCM16_VERIFICADOS',len(rows),'duracion_audio_s',sum(r['duration_s'] for r in rows))
"""
(run/'convert.py').write_text(convert,encoding='utf-8')
subprocess.run(['/content/dictalibre_venv/bin/python',str(run/'convert.py'),str(run)],check=True)
print('VOZ_A_LISTA',run.name)
for row in json.loads((run/'conversion.json').read_text()):
    print(row['case_id'],round(row['review_start_s'],2),row['text'])
display(Audio(filename=str(run/'escucha_12_A.wav'),normalize=False))
archive=shutil.make_archive('/content/dictalibre_voz_A_12_'+run.name,'zip',run)
print('ZIP',archive)
files.download(archive)
